# Stage 6 — Scoring, Ranking, and Statistical Testing

Reads the per-run CSVs produced by Stages 3–5 and produces:

- Aggregated rankings (mean ± std across seeds) for both NLP and
  regression models.
- A one-way ANOVA on NLP F1 across models, plus Tukey HSD pairwise
  p-values.
- Diagnostic plots for the NLP F1 distribution and the Tukey heatmap.

Once energy columns are present, swap `"f1"` / `"r2"` below for a
composite energy-efficiency score.


In [ ]:
# Shared paths/config plus pandas/numpy imports from common.py.
from common import *

# Plotting and statistical testing libraries used in this stage.
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Apply a clean plotting theme for all diagnostic charts.
sns.set_theme(style="whitegrid")

# Load run-level metrics produced by earlier stages.
nlp_df = pd.read_csv(RESULTS_DIR / "nlp_results.csv")
reg_df = pd.read_csv(RESULTS_DIR / "regression_results.csv")
print(f"Loaded {len(nlp_df)} NLP rows, {len(reg_df)} regression rows")

## 6.1 Aggregate and rank

In [ ]:
def score_and_rank(df, accuracy_col):
    """Aggregate accuracy and timing across seeds, then rank models."""
    agg = df.groupby("model").agg(
        accuracy_mean=(accuracy_col, "mean"),
        accuracy_std=(accuracy_col, "std"),
        train_time_mean=("train_time_s", "mean"),
        infer_time_mean=("infer_time_s", "mean"),
    ).reset_index()
    return agg


nlp_scores = score_and_rank(nlp_df, "f1")
print("=== NLP Model Rankings ===")
display(nlp_scores.sort_values("accuracy_mean", ascending=False).round(6))

reg_scores = score_and_rank(reg_df, "r2")
print("\n=== Regression Model Rankings (baseline + NLP-augmented) ===")
display(reg_scores.sort_values("accuracy_mean", ascending=False).round(6))

## 6.2 ANOVA + Tukey HSD on NLP F1

In [ ]:
# model_order fixes display/statistics order across outputs.
model_order = sorted(nlp_df["model"].unique())
# groups is one F1-score array per model for ANOVA/Tukey testing.
groups = [nlp_df.loc[nlp_df["model"] == m, "f1"].values for m in model_order]

# ANOVA tests whether at least one model mean differs.
anova_result = stats.f_oneway(*groups)
print("=== One-way ANOVA on NLP F1 ===")
print(f"F-statistic: {anova_result.statistic:.6f}")
print(f"p-value:     {anova_result.pvalue:.6e}")

try:
    # Tukey HSD performs pairwise post-hoc significance tests.
    tukey_result = stats.tukey_hsd(*groups)
    tukey_pvalues = pd.DataFrame(
        tukey_result.pvalue, index=model_order, columns=model_order
    )
    print("\n=== Tukey HSD pairwise p-values ===")
    display(tukey_pvalues.round(6))
except Exception as e:
    tukey_pvalues = None
    print(f"\nTukey HSD unavailable in this scipy version: {e}")

## 6.3 Diagnostic plots

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# 1) Per-seed F1 distribution
sns.boxplot(data=nlp_df, x="model", y="f1", order=model_order, ax=axes[0])
sns.stripplot(data=nlp_df, x="model", y="f1", order=model_order,
              color="black", alpha=0.45, size=4, ax=axes[0])
axes[0].set_title(f"NLP Model F1 Distribution ({nlp_df['seed'].nunique()} seeds)")
axes[0].set_xlabel("Model")
axes[0].set_ylabel("F1")
axes[0].tick_params(axis="x", rotation=25)

# 2) Mean ± std bar chart
nlp_summary = nlp_df.groupby("model")["f1"].agg(["mean", "std"]).reindex(model_order)
axes[1].bar(nlp_summary.index, nlp_summary["mean"],
            yerr=nlp_summary["std"], capsize=5)
axes[1].set_title("NLP Mean F1 with Std Dev")
axes[1].set_xlabel("Model")
axes[1].set_ylabel("Mean F1")
axes[1].tick_params(axis="x", rotation=25)

# 3) Tukey HSD p-value heatmap
if tukey_pvalues is not None:
    sns.heatmap(
        tukey_pvalues.loc[model_order, model_order],
        annot=True, fmt=".3f", cmap="viridis_r",
        cbar_kws={"label": "p-value"},
        ax=axes[2],
    )
    axes[2].set_title("Tukey HSD Pairwise p-values")
else:
    axes[2].text(0.5, 0.5, "Tukey HSD unavailable", ha="center", va="center")
    axes[2].set_axis_off()

plt.tight_layout()
plt.savefig(RESULTS_DIR / "nlp_f1_diagnostics.png", dpi=150, bbox_inches="tight")
plt.show()

## 6.4 Regression uplift table (baseline vs +NLP)

In [ ]:
baseline_r2 = (
    reg_df[~reg_df["model"].str.endswith("+NLP")]
    .groupby("model")["r2"]
    .mean()
    .rename("baseline_r2")
)

augmented_r2 = (
    reg_df[reg_df["model"].str.endswith("+NLP")]
    .assign(base_model=lambda frame: frame["model"].str.replace(" +NLP", "", regex=False))
    .groupby("base_model")["r2"]
    .mean()
    .rename("augmented_r2")
)

uplift_table = pd.concat([baseline_r2, augmented_r2], axis=1)
uplift_table["delta"] = uplift_table["augmented_r2"] - uplift_table["baseline_r2"]

print("=== Regression uplift from real NLP features (mean over seeds) ===")
display(uplift_table.sort_values("delta", ascending=False).round(6))